# 1 – Vorhandenes Plugin nutzen: GitLab Catalog Discovery

Dieses Notebook installiert das offizielle GitLab-Catalog-Backend-Modul. Es durchsucht eine GitLab-Gruppe nach `catalog-info.yaml`-Dateien und registriert die gefundenen Entities im Backstage Catalog.


> **Voraussetzung:** Die Backstage-Installation liegt unter `~/mybackstage`.
>
> Shell-Zellen werden über Python mit `subprocess` ausgeführt. Vor Änderungen wird jeweils eine Sicherung angelegt.


Benötigt werden:

- eine GitLab-Gruppe oder ein GitLab-Namensraum,
- ein GitLab Access Token mit mindestens `read_api`,
- `catalog-info.yaml` in den gewünschten Repositories.

In [ ]:
from pathlib import Path
ROOT = Path.home() / "mybackstage"
assert ROOT.exists(), f"{ROOT} wurde nicht gefunden"
print("Backstage:", ROOT)
print((ROOT / "package.json").read_text()[:500])

## Paket installieren

In [ ]:
import subprocess
subprocess.run(
    ["yarn", "--cwd", "packages/backend", "add",
     "@backstage/plugin-catalog-backend-module-gitlab"],
    cwd=ROOT,
    check=True,
)

## Backend-Modul registrieren

Die folgende Zelle ergänzt `packages/backend/src/index.ts` nur dann, wenn der Import noch nicht vorhanden ist.

In [ ]:
from pathlib import Path
import shutil, datetime

index_file = ROOT / "packages/backend/src/index.ts"
backup = index_file.with_suffix(f".ts.bak-{datetime.datetime.now():%Y%m%d-%H%M%S}")
shutil.copy2(index_file, backup)

text = index_file.read_text()
line = "backend.add(import('@backstage/plugin-catalog-backend-module-gitlab'));"

if line not in text:
    anchor = "backend.start();"
    if anchor not in text:
        raise RuntimeError("backend.start(); wurde in packages/backend/src/index.ts nicht gefunden")
    text = text.replace(anchor, f"{line}\n\n{anchor}")
    index_file.write_text(text)
    print("GitLab-Modul ergänzt.")
else:
    print("GitLab-Modul ist bereits registriert.")

print("Sicherung:", backup)

## GitLab-Konfiguration ergänzen

Passe `host`, `group` und bei Bedarf den Provider-Namen an. Das Token wird nicht in die YAML-Datei geschrieben, sondern über die Umgebungsvariable `GITLAB_TOKEN` gelesen.

In [ ]:
from pathlib import Path
import shutil, datetime

config_file = ROOT / "app-config.local.yaml"
if not config_file.exists():
    config_file.write_text("# Lokale, nicht versionierte Konfiguration\n")

backup = config_file.with_suffix(f".yaml.bak-{datetime.datetime.now():%Y%m%d-%H%M%S}")
shutil.copy2(config_file, backup)

block = """
integrations:
  gitlab:
    - host: gitlab.com
      token: ${GITLAB_TOKEN}

catalog:
  providers:
    gitlab:
      training:
        host: gitlab.com
        group: DEINE-GITLAB-GRUPPE
        branch: main
        entityFilename: catalog-info.yaml
        skipForkedRepos: true
        includeArchivedRepos: false
        schedule:
          frequency: { minutes: 30 }
          timeout: { minutes: 10 }
"""

text = config_file.read_text()
if "providers:\n    gitlab:" not in text:
    with config_file.open("a") as f:
        f.write("\n" + block.strip() + "\n")
    print("Konfigurationsblock ergänzt:", config_file)
else:
    print("Eine GitLab-Provider-Konfiguration ist bereits vorhanden. Bitte manuell prüfen.")

print("Sicherung:", backup)

## Token setzen und Konfiguration prüfen

In [ ]:
import os, subprocess, getpass

if not os.environ.get("GITLAB_TOKEN"):
    os.environ["GITLAB_TOKEN"] = getpass.getpass("GitLab Token: ")

subprocess.run(
    ["yarn", "backstage-cli", "config:print",
     "--config", "app-config.yaml",
     "--config", "app-config.local.yaml"],
    cwd=ROOT,
    env=os.environ,
    check=True,
)

## Backstage starten

In [ ]:
import subprocess, os
print("Start in einem separaten Terminal:")
print(f"cd {ROOT} && export GITLAB_TOKEN='***' && yarn start")

## Kontrolle

Nach dem Start:

1. Catalog öffnen.
2. Prüfen, ob Components aus der konfigurierten GitLab-Gruppe erscheinen.
3. Backend-Log nach `GitlabDiscoveryEntityProvider` durchsuchen.

Ein minimales Repository benötigt beispielsweise:

```yaml
apiVersion: backstage.io/v1alpha1
kind: Component
metadata:
  name: example-service
spec:
  type: service
  lifecycle: experimental
  owner: user:default/guest
```